In [ ]:
# # !pip install mysql-connector-python
# !pip uninstall -y mysql-connector mysql-connector-python
# !pip install mysql-connector-python==8.0.33


In [1]:
import pandas
import os
import mysql.connector
import json
from datetime import datetime
from mysql.connector import Error



In [3]:
try:
    conn = mysql.connector.connect(
        host="127.0.0.1",
        port=3306,
        user="your_username",
        password="your_password",
        database="cric_data",
        ssl_disabled=True
    )

    cursor = conn.cursor()
    print("✅ MySQL connection established successfully.")

except Error as e:
    print("❌ Error while connecting to MySQL:")
    print(f"Error Code: {e.errno}")
    print(f"SQLSTATE: {e.sqlstate}")
    print(f"Message: {e.msg}")

✅ MySQL connection established successfully.


In [5]:

DATA_DIR = r"D:\your_path"

for filename in os.listdir(DATA_DIR):
    if filename.endswith(".json"):
        try:
            filepath = os.path.join(DATA_DIR, filename)
            with open(filepath, "r") as file:
                data = json.load(file)
    
            match_id = filename.replace(".json", "")
    
             # ========== TEAMS TABLE ==========

            current_section = "Insert TEAMS"
            
            all_team_names = data.get("info", {}).get("teams")
    
            for ind_team in all_team_names:
                team_id = '.'.join([x[0] for x in ind_team.split(" ")]) + "-" + "Team"
                cursor.execute("""
                    INSERT IGNORE INTO teams (team_id, team_name)
                    VALUES (%s, %s)
                """, (team_id, ind_team))
    
                print("Inserted TEAMS TABLE")
    
            # ========== PLAYERS TABLE ==========

            current_section = "Insert PLAYERS"
            
            for ind_team in all_team_names:
                team_name = ind_team
                all_player_names = data.get("info", {}).get("players", {}).get(ind_team)
                for ind_player in all_player_names:
                    player_id = data.get("info", {}).get("registry", {}).get("people", {}).get(ind_player)
                    player_name = ind_player
                    # print(team_name, player_name, player_id)
    
                    cursor.execute("""
                        INSERT IGNORE INTO players (
                            player_id, player_name, team_name
                        ) VALUES (%s, %s, %s)
                    """, (player_id, player_name, team_name))

            print("Inserted PLAYERS TABLE")
    
            # ========== MATCH DETAILS TABLE ==========
    

            current_section = "Insert MATCH DETAILS"
            
            dates = data.get('info', {}).get('dates', [])
            match_date = dates[0] if dates else None
            city = data.get('info', {}).get('city')
            venue = data.get('info', {}).get('venue')
            match_number = int(data.get('info', {}).get('event', {}).get('match_number', 0))
            match_type = data.get('info', {}).get('match_type')
            gender = data.get('info', {}).get('gender', '').lower()
            event_name = data.get('info', {}).get('event', {}).get('name')
            balls_per_over = int(data.get('info', {}).get('balls_per_over', 6))
            overs = int(data.get('info', {}).get('overs', 20))
            season = data.get('info', {}).get('season')
            team_type = data.get('info', {}).get('team_type')
            teams = data.get('info', {}).get('teams', [None, None])
            team_1 = teams[0].strip() if teams[0] else None
            team_2 = teams[1].strip() if teams[1] else None
            toss_winner_team = data.get('info', {}).get('toss', {}).get('winner')
            toss_decision = data.get('info', {}).get('toss', {}).get('decision')
            winner_team = data.get('info', {}).get('outcome', {}).get('winner')
            winner_team = winner_team.strip() if winner_team else None
            win_by_runs = data.get('info', {}).get('outcome', {}).get('by', {}).get('runs')
            win_by_wickets = data.get('info', {}).get('outcome', {}).get('by', {}).get('wickets')
            player_of_match_list = data.get('info', {}).get('player_of_match', [])
            player_of_match = player_of_match_list[0] if player_of_match_list else None
            player_of_match_id = data.get("info", {}).get("registry", {}).get("people", {}).get(player_of_match)
            stage = data.get('info', {}).get('event', {}).get('stage')

            print(f"⚠️ Match {match_id} → team_1: {team_1}, team_2: {team_2}")
            print(f"🏏 Match {match_id}")
            print(f"team_1 → {team_1}")
            print(f"team_2 → {team_2}")
            print(f"match_number → {match_number}")
            print(f"winner_team → {winner_team}")
            print(f"balls_per_over → {balls_per_over}")
            print(f"overs → {overs}")

    
            cursor.execute("""
                        INSERT INTO match_detail (
            match_id, match_date, city, venue, match_number, stage, match_type, gender, 
            event_name, balls_per_over, overs, season, team_type,
            team_1, team_2, toss_winner_team, toss_decision, winner_team,
            win_by_runs, win_by_wickets, player_of_match_id
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    """,  (match_id, match_date, city, venue, match_number, stage, match_type, gender, 
                         event_name, balls_per_over, overs, season, team_type,
                         team_1, team_2, toss_winner_team, toss_decision, winner_team,
                         win_by_runs, win_by_wickets, player_of_match_id))
            
            print("Inserted MATCH DETAILS TABLE")

             # ========== MATCH PLAYERS TABLE ==========

            current_section = "Insert MATCH PLAYER"
            
            for ind_team in all_team_names:
                team_name = ind_team
                all_player_names = data.get("info", {}).get("players", {}).get(ind_team)
                for ind_player in all_player_names:
                    player_id = data.get("info", {}).get("registry", {}).get("people", {}).get(ind_player)
                    player_name = ind_player

                    cursor.execute("""
                        INSERT INTO match_players (
                            match_id, player_id, team_name
                        ) VALUES (%s, %s, %s)
                    """, (
                        match_id, player_id, team_name
                    ))

            print("Inserted MATCH PLAYERS TABLE")
    
            # ========== OFFICIALS TABLE ==========

            current_section = "Insert OFFICIALS"
    
            all_officials = data.get('info', {}).get('officials')
            all_officials_name = []
            for ind_off in all_officials:
                all_officials_name.extend(all_officials[ind_off])
            
            for ind_off in all_officials_name:
                official_id = data.get("info", {}).get("registry", {}).get("people", {}).get(ind_off)
                official_name = ind_off
   
                cursor.execute("""
                INSERT IGNORE INTO officials (
                    official_id,
                    official_name
                ) VALUES (%s, %s)
                """, (official_id, official_name))

            print("Inserted OFFICIALS TABLE")
    
    
            # ========== DELIVERIES TABLE ==========

            current_section = "Insert DELIVERIES"
    
            teams = data.get('info', {}).get('teams', [None, None])
            
            innings = data.get('innings')
            inning_num = 0
            for inning in innings:
                
                inning_num = inning_num + 1
                batting_team = inning.get('team')
                bowling_team_set = set(data.get('info', {}).get('teams', [None, None]))
                bowling_team_set.discard(batting_team)
                bowling_team_list = list(bowling_team_set)
                bowling_team = bowling_team_list[0] if bowling_team_list else None
                
                all_overs = inning['overs']
                for delivery in all_overs:
                    over_num = int(delivery.get('over'))
                    ball_num = 0
                    all_deliveries = delivery.get('deliveries')
                    for ind_delivery in all_deliveries:
                        
                        ball_num = ball_num + 1
                        
                        batsman = ind_delivery.get('batter')
                        batsman_id = data.get("info", {}).get("registry", {}).get("people", {}).get(batsman)
                        bowler = ind_delivery.get('bowler')
                        bowler_id = data.get("info", {}).get("registry", {}).get("people", {}).get(bowler)
                        non_striker = ind_delivery.get('non_striker')
                        non_striker_id = data.get("info", {}).get("registry", {}).get("people", {}).get(non_striker)
    
                        avv = '_vs_'.join([''.join(word[0] for word in team.split()) for team in teams])
                        delivery_id = str(avv) + "_" + str(match_id) + "_" + str(over_num) + "_" + str(ball_num) + "_" + str(inning_num)

                        runs_batsman = int(ind_delivery.get('runs', {}).get('batter'))
                        runs_extras = int(ind_delivery.get('runs', {}).get('extras'))
                        runs_total = int(ind_delivery.get('runs', {}).get('total'))
                        extras = ind_delivery.get('extras', {})
                        extras_type = list(extras.keys())[0] if extras else None
                        extras_runs = int(extras.get(extras_type, 0)) if extras_type else 0
                        
                        # extras_type = list(ind_delivery.get('extras').keys())[0] if extras else None
                        # extras_runs = int(ind_delivery.get('extras').get(extras_type)) if extras_type else 0
                        
            
                        cursor.execute("""
                            INSERT INTO deliveries (
                                delivery_id, match_id, inning_num, batting_team, bowling_team,
                                over_num, ball_num, batsman_id, bowler_id, non_striker_id,
                                runs_batsman, runs_extras, runs_total, extras_type, extras_runs
                            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                        """, (
                            delivery_id, match_id, inning_num, batting_team, bowling_team,
                            over_num, ball_num, batsman_id, bowler_id, non_striker_id,
                            runs_batsman, runs_extras, runs_total, extras_type, extras_runs
                        ))
                        
                         # ========== REVIEWS TABLE ==========

                        current_section = "Insert REVIEWS"
                                
                        if ind_delivery.get('review', {}).get('by'):
                            review_id = delivery_id + "_REVIEW"
                        
                            review_by_team = ind_delivery.get('review', {}).get('by')
                            umpire_name = ind_delivery.get('review', {}).get('umpire')
                            umpire_id = data.get("info", {}).get("registry", {}).get("people", {}).get(umpire_name)
                            batsman_name = ind_delivery.get('review', {}).get('batter')
                            batsman_name_id = data.get("info", {}).get("registry", {}).get("people", {}).get(batsman_name)
                            decision = ind_delivery.get('review', {}).get('decision')
                            review_type = ind_delivery.get('review', {}).get('type')
                            umpires_call = ind_delivery.get('review', {}).get('umpires_call')
                        
                            cursor.execute("""
                                INSERT INTO reviews (
                                    review_id,
                                    match_id,
                                    delivery_id,
                                    review_by_team,
                                    umpire_id,
                                    batsman_name_id,
                                    decision,
                                    review_type,
                                    umpires_call
                                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                            """, (
                                review_id,
                                match_id,
                                delivery_id,
                                review_by_team,
                                umpire_id,
                                batsman_name_id,
                                decision,
                                review_type,
                                umpires_call
                            ))
                        
                        
                        # ========== REPLACEMENT TABLE ==========
                        
                        current_section = "Insert REPLACEMENT"
                        
                        if ind_delivery.get('replacements', {}).get('match'):
                            all_replacement_player = ind_delivery.get('replacements', {}).get('match')
                            for i, ind_rep_player in enumerate(all_replacement_player):
                                replacement_id = f"{delivery_id}_REPL_{i+1}"
                        
                                rep_team_name = ind_rep_player.get('team')
                                player_in_name = ind_rep_player.get('in')
                                player_in_id = data.get("info", {}).get("registry", {}).get("people", {}).get(player_in_name)
                                player_out_name = ind_rep_player.get('out')
                                player_out_id = data.get("info", {}).get("registry", {}).get("people", {}).get(player_out_name)
                                reason = ind_rep_player.get('reason')
                        
                                cursor.execute("""
                                    INSERT INTO replacements (
                                        replacement_id,
                                        match_id,
                                        team_name,
                                        player_in_id,
                                        player_out_id,
                                        reason
                                    ) VALUES (%s, %s, %s, %s, %s, %s)
                                """, (
                                    replacement_id,
                                    match_id,
                                    rep_team_name,
                                    player_in_id,
                                    player_out_id,
                                    reason
                                ))
                        
                        
                        # ========== WICKET TABLE ==========
                        
                        current_section = "Insert WICKET"
                        
                        if ind_delivery.get('wickets'):
                            all_wickets = ind_delivery.get('wickets')
                            for i, wkt in enumerate(all_wickets):
                                wicket_id_base = f"{delivery_id}_WKT_{i+1}"
                        
                                player_dismissed = wkt.get("player_out")
                                player_dismissed_id = data.get("info", {}).get("registry", {}).get("people", {}).get(player_dismissed)
                                dismissal_kind = wkt.get('kind')
                                all_fielder = wkt.get('fielders')
                        
                                if not all_fielder:
                                    fielder_id = None
                                    cursor.execute("""
                                        INSERT INTO wickets (
                                            wicket_id,
                                            delivery_id,
                                            player_dismissed_id,
                                            dismissal_kind,
                                            fielder_id
                                        ) VALUES (%s, %s, %s, %s, %s)
                                    """, (
                                        wicket_id_base,
                                        delivery_id,
                                        player_dismissed_id,
                                        dismissal_kind,
                                        fielder_id
                                    ))
                                else:
                                    for j, ind_fielder in enumerate(all_fielder):
                                        fielder = ind_fielder.get('name')
                                        fielder_id = data.get("info", {}).get("registry", {}).get("people", {}).get(fielder)
                        
                                        # Handle substitute fielder
                                        if ind_fielder.get("substitute", False):
                                            cursor.execute("""
                                                INSERT IGNORE INTO players (player_id, player_name, team_name)
                                                VALUES (%s, %s, %s)
                                            """, (fielder_id, fielder, "Substitute"))
                        
                                        full_wicket_id = f"{wicket_id_base}_{j+1}"
                        
                                        cursor.execute("""
                                            INSERT INTO wickets (
                                                wicket_id,
                                                delivery_id,
                                                player_dismissed_id,
                                                dismissal_kind,
                                                fielder_id
                                            ) VALUES (%s, %s, %s, %s, %s)
                                        """, (
                                            full_wicket_id,
                                            delivery_id,
                                            player_dismissed_id,
                                            dismissal_kind,
                                            fielder_id
                                        ))


            print("Inserted DELIVERIES, REVIEWS, REPLACEMENR & WICKET TABLE")

            # ========== OFFICIAL ROLE TABLE ==========

            current_section = "Insert OFFICIAL ROLE"
            
            all_officials = data.get('info', {}).get('officials')
            for ind_official_role in all_officials:
                roles = ind_official_role
                for off_name in all_officials[ind_official_role]:
                    official_id = data.get("info", {}).get("registry", {}).get("people", {}).get(off_name)
                    cursor.execute("""
                        INSERT INTO match_officials (
                            match_id,
                            official_id,
                            roles
                        ) VALUES (%s, %s, %s)
                        """, (
                        match_id,
                        official_id,
                        roles
                    ))

            print("Inserted OFFICIAL ROLE TABLE")

            conn.commit()
            print(f"✅ Inserted {filename}")
            
        except mysql.connector.Error as e:
            conn.rollback()
            print(f"❌ MySQL Error in file {filename} → Section: {current_section} → {e}")
        
        except Exception as e:
            conn.rollback()
            print(f"❌ General Error in file {filename} → Section: {current_section} → {e}")

conn.close()

print("✅ Matches and match_meta tables populated!")

Inserted TEAMS TABLE
Inserted TEAMS TABLE
Inserted PLAYERS TABLE
⚠️ Match 1473438 → team_1: Kolkata Knight Riders, team_2: Royal Challengers Bengaluru
🏏 Match 1473438
team_1 → Kolkata Knight Riders
team_2 → Royal Challengers Bengaluru
match_number → 1
winner_team → Royal Challengers Bengaluru
balls_per_over → 6
overs → 20
Inserted MATCH DETAILS TABLE
Inserted MATCH PLAYERS TABLE
Inserted OFFICIALS TABLE
Inserted DELIVERIES, REVIEWS, REPLACEMENR & WICKET TABLE
Inserted OFFICIAL ROLE TABLE
✅ Inserted 1473438.json
Inserted TEAMS TABLE
Inserted TEAMS TABLE
Inserted PLAYERS TABLE
⚠️ Match 1473439 → team_1: Sunrisers Hyderabad, team_2: Rajasthan Royals
🏏 Match 1473439
team_1 → Sunrisers Hyderabad
team_2 → Rajasthan Royals
match_number → 2
winner_team → Sunrisers Hyderabad
balls_per_over → 6
overs → 20
Inserted MATCH DETAILS TABLE
Inserted MATCH PLAYERS TABLE
Inserted OFFICIALS TABLE
Inserted DELIVERIES, REVIEWS, REPLACEMENR & WICKET TABLE
Inserted OFFICIAL ROLE TABLE
✅ Inserted 1473439.json


In [ ]:
conn.close()